# Invoke model 

https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards.html



In [42]:
import sys
import boto3
import json



In [43]:
# List Amazon Bedrock Foundation Models
bedrock = boto3.client('bedrock')
bedrock_runtime = boto3.client('bedrock-runtime')

# List available foundation models
response = bedrock.list_foundation_models()

print("Available Foundation Models:")
print("-" * 50)
for model in response['modelSummaries']:
    print(f"Model ID: {model['modelId']}")
    print(f"  Provider: {model['providerName']}")
    print(f"  Name: {model['modelName']}")
    print(f"  Input Modalities: {model['inputModalities']}")
    print(f"  Output Modalities: {model['outputModalities']}")
    print("-" * 50)

Available Foundation Models:
--------------------------------------------------
Model ID: google.gemma-3-4b-it
  Provider: Google
  Name: Gemma 3 4B IT
  Input Modalities: ['TEXT', 'IMAGE']
  Output Modalities: ['TEXT']
--------------------------------------------------
Model ID: nvidia.nemotron-nano-12b-v2
  Provider: NVIDIA
  Name: NVIDIA Nemotron Nano 12B v2 VL BF16
  Input Modalities: ['TEXT', 'IMAGE']
  Output Modalities: ['TEXT']
--------------------------------------------------
Model ID: anthropic.claude-opus-4-8
  Provider: Anthropic
  Name: Claude Opus 4.8
  Input Modalities: ['TEXT', 'IMAGE']
  Output Modalities: ['TEXT']
--------------------------------------------------
Model ID: anthropic.claude-opus-4-7
  Provider: Anthropic
  Name: Claude Opus 4.7
  Input Modalities: ['TEXT', 'IMAGE']
  Output Modalities: ['TEXT']
--------------------------------------------------
Model ID: anthropic.claude-haiku-4-5-20251001-v1:0
  Provider: Anthropic
  Name: Claude Haiku 4.5
  Input M

In [ ]:
# Standard Prompt for testing multiple models
standard_prompt = """Write a short paragraph about the benefits of artificial intelligence in healthcare."""

print("Standard Prompt:")
print(standard_prompt)
print("\n" + "=" * 80 + "\n")

# Initialize Bedrock clients
bedrock = boto3.client('bedrock')
bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

Standard Prompt:
Write a short paragraph about the benefits of artificial intelligence in healthcare.




### Nova pro model
https://docs.aws.amazon.com/nova/latest/userguide/using-invoke-api.html


In [67]:
model_id = "us.amazon.nova-pro-v1:0"
body = json.dumps({
    "schemaVersion": "messages-v1",
    "messages": [
        {
            "role": "user",
            "content": [{"text": standard_prompt}]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 512,
        "temperature": 0.7,
        "topP": 0.9
    }
})
response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=body
    )

    # Read and parse the response
response_body = json.loads(response['body'].read())
    
    # 3. The output structure is also different for the messages schema
generated_text = response_body['output']['message']['content'][0]['text']

print(model_id)
print("Input API:")
print(json.dumps(json.loads(body), indent=2))
print("\nOutput API (Truncated for readability):")
print(json.dumps(response_body['usage'], indent=2)) 
print("\nGenerated Text:")
print(generated_text)
print("-" * 80)



us.amazon.nova-pro-v1:0
Input API:
{
  "schemaVersion": "messages-v1",
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "text": "Write a short paragraph about the benefits of artificial intelligence in healthcare."
        }
      ]
    }
  ],
  "inferenceConfig": {
    "maxTokens": 512,
    "temperature": 0.7,
    "topP": 0.9
  }
}

Output API (Truncated for readability):
{
  "inputTokens": 13,
  "outputTokens": 105,
  "totalTokens": 118,
  "cacheReadInputTokenCount": 0,
  "cacheWriteInputTokenCount": 0
}

Generated Text:
Artificial Intelligence (AI) is revolutionizing healthcare by enhancing diagnostic accuracy, personalizing treatment plans, and improving patient outcomes. AI algorithms can analyze vast amounts of medical data rapidly, identifying patterns and predicting disease outbreaks, which aids in early intervention. Additionally, AI-driven tools facilitate administrative efficiency, reducing paperwork and allowing healthcare professionals to 

### qwen3

In [55]:
qwen_model_id = "qwen.qwen3-32b-v1:0"
print(f"Testing Model: {qwen_model_id}")

# Qwen 3 on Bedrock now expects an OpenAI-style 'messages' array at the root
qwen_body = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": standard_prompt
        }
    ],
    "max_tokens": 512,
    "temperature": 0.7,
    "top_p": 0.9
})

try:
    response = bedrock_runtime.invoke_model(modelId=qwen_model_id, body=qwen_body)
    response_body = json.loads(response['body'].read())
    
    # Because it uses the messages schema, the output is parsed differently 
    qwen_text = response_body['choices'][0]['message']['content']
    
    print("\nGenerated Text:\n", qwen_text.strip())
except Exception as e:
    print(f"Error invoking Qwen: {e}")

print("-" * 80)

Testing Model: qwen.qwen3-32b-v1:0

Generated Text:
 Artificial intelligence (AI) is revolutionizing healthcare by enhancing diagnostic accuracy, streamlining administrative tasks, and personalizing patient care. AI algorithms can analyze vast amounts of medical data quickly, enabling early detection of diseases such as cancer and heart conditions. Additionally, AI-powered tools assist in drug discovery, reduce human error, and improve treatment planning. By automating routine processes, healthcare professionals can focus more on patient care, ultimately leading to better health outcomes and increased efficiency in the medical field.
--------------------------------------------------------------------------------



### llama3

In [59]:
llama_model_id = "meta.llama3-8b-instruct-v1:0"
print(f"Testing Model: {llama_model_id}")

# Llama uses 'max_gen_len' instead of max_tokens and requires specific header tags
llama_body = json.dumps({
    "prompt": f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{standard_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
    "max_gen_len": 512,
    "temperature": 0.7,
    "top_p": 0.9
})

try:
    response = bedrock_runtime.invoke_model(modelId=llama_model_id, body=llama_body)
    response_body = json.loads(response['body'].read())
    
    # Llama returns text directly under the 'generation' key
    llama_text = response_body['generation']
    
    #print("Input Payload Keys : 'prompt', 'max_gen_len', 'temperature'")
    #print("Output JSON Path   : response_body['generation']")
    print("\nGenerated Text:\n", llama_text.strip())
except Exception as e:
    print(f"Error invoking Llama: {e}")

print("-" * 80)

Testing Model: meta.llama3-8b-instruct-v1:0

Generated Text:
 The integration of artificial intelligence (AI) in healthcare has revolutionized the industry by providing numerous benefits. AI-powered systems have improved diagnostic accuracy, enabling healthcare professionals to make more informed decisions and reduce misdiagnosis rates. Additionally, AI-assisted patient monitoring systems have enabled early detection of potential health issues, allowing for prompt interventions and improved patient outcomes. AI-powered chatbots and virtual assistants have also enhanced patient engagement, providing personalized support and education to patients. Furthermore, AI-driven data analytics have optimized treatment plans, reduced healthcare costs, and improved resource allocation. Overall, AI has transformed the healthcare landscape, enabling more efficient, effective, and personalized care for patients.
--------------------------------------------------------------------------------


### Google Gemma 3 

In [66]:
gemma_model_id = "google.gemma-3-4b-it" 
print(f"Testing Model: {gemma_model_id}")
print("-" * 80)

# Gemma 3 on Bedrock now requires the 'messages' array, NOT a raw string prompt
gemma_body = json.dumps({
    "messages": [
        {
            "role": "user",
            "content": standard_prompt
        }
    ],
    "max_tokens": 512,
    "temperature": 0.7,
    "top_p": 0.9
})

try:
    response = bedrock_runtime.invoke_model(modelId=gemma_model_id, body=gemma_body)
    response_body = json.loads(response['body'].read())
    
    # Because it uses the messages API schema, text is typically found here
    if 'choices' in response_body:
        gemma_text = response_body['choices'][0]['message']['content']
    elif 'outputs' in response_body:
        gemma_text = response_body['outputs'][0].get('text', '')
    else:
        gemma_text = "Could not parse text. Raw output: " + str(response_body)
    
    print("\nGenerated Text:\n", gemma_text.strip())

except Exception as e:
    print(f"Error invoking Gemma 3: {e}")

print("-" * 80)

Testing Model: google.gemma-3-4b-it
--------------------------------------------------------------------------------

Generated Text:
 Artificial intelligence is rapidly transforming healthcare, offering a multitude of benefits. From accelerating drug discovery and personalizing treatment plans based on individual patient data to improving diagnostic accuracy through image analysis and streamlining administrative tasks, AI is poised to revolutionize the industry. It can also empower patients with wearable technology that monitors their health and provides proactive alerts, ultimately leading to earlier diagnoses, more effective treatments, and a more efficient and accessible healthcare system for everyone.
--------------------------------------------------------------------------------


# Converse API

In [68]:
import boto3
import json

# Initialize the Bedrock Runtime client
bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

# Standard Prompt
standard_prompt = "Write a short paragraph about the benefits of artificial intelligence in healthcare."

# Standard Message Structure for the Converse API
messages = [
    {
        "role": "user",
        "content": [{"text": standard_prompt}]
    }
]

# Standard Inference Parameters
inference_config = {
    "maxTokens": 512,
    "temperature": 0.7,
    "topP": 0.9
}

### Amazon Nova Pro

In [69]:
model_id = "us.amazon.nova-pro-v1:0"
print(f"Calling Model: {model_id}\n" + "-" * 40)

try:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=messages,
        inferenceConfig=inference_config
    )
    
    generated_text = response['output']['message']['content'][0]['text']
    print(generated_text.strip())

except Exception as e:
    print(f"Error: {e}")

Calling Model: us.amazon.nova-pro-v1:0
----------------------------------------
Artificial Intelligence (AI) has revolutionized healthcare by enhancing diagnostic accuracy, personalizing treatment plans, and optimizing operational efficiencies. AI algorithms can analyze complex medical data rapidly, identifying patterns and predicting patient outcomes with greater precision than traditional methods. This leads to earlier detection of diseases, tailored therapies based on genetic profiles, and improved patient care. Additionally, AI-driven automation streamlines administrative tasks, reducing costs and allowing healthcare professionals to focus more on patient interaction and complex medical cases. Overall, the integration of AI in healthcare promises to elevate the quality of care, improve patient satisfaction, and foster innovative medical solutions.


### Qwen 3 (32B)

In [70]:
model_id = "qwen.qwen3-32b-v1:0"
print(f"Calling Model: {model_id}\n" + "-" * 40)

try:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=messages,
        inferenceConfig=inference_config
    )
    
    generated_text = response['output']['message']['content'][0]['text']
    print(generated_text.strip())

except Exception as e:
    print(f"Error: {e}")

Calling Model: qwen.qwen3-32b-v1:0
----------------------------------------
Artificial intelligence (AI) is transforming healthcare by improving diagnostic accuracy, streamlining operations, and personalizing patient care. AI algorithms can analyze vast amounts of medical data quickly, enabling early detection of diseases such as cancer and diabetes. It also enhances treatment planning by tailoring therapies to individual patient needs. Additionally, AI supports administrative efficiency through automated scheduling and medical record management, reducing the burden on healthcare professionals. By augmenting human capabilities, AI ultimately contributes to better health outcomes and a more efficient healthcare system.


### Meta Llama 3 (8B Instruct)

In [71]:
model_id = "meta.llama3-8b-instruct-v1:0"
print(f"Calling Model: {model_id}\n" + "-" * 40)

try:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=messages,
        inferenceConfig=inference_config
    )
    
    generated_text = response['output']['message']['content'][0]['text']
    print(generated_text.strip())

except Exception as e:
    print(f"Error: {e}")

Calling Model: meta.llama3-8b-instruct-v1:0
----------------------------------------
The integration of artificial intelligence (AI) in healthcare has revolutionized the industry by providing numerous benefits. AI-powered systems can analyze vast amounts of medical data to identify patterns and predict patient outcomes, enabling healthcare providers to make more informed decisions. AI-assisted diagnostics can detect diseases earlier and more accurately, leading to improved patient outcomes and reduced healthcare costs. Additionally, AI-powered chatbots and virtual assistants can help streamline patient communication, freeing up healthcare professionals to focus on higher-value tasks. Furthermore, AI-powered robots can assist with surgeries, reducing recovery times and minimizing human error. Overall, the incorporation of AI in healthcare has the potential to improve patient care, increase efficiency, and reduce costs, ultimately leading to a more effective and sustainable healthcare sy

### Google Gemma 3 (4B IT)

In [72]:
model_id = "google.gemma-3-4b-it"
print(f"Calling Model: {model_id}\n" + "-" * 40)

try:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=messages,
        inferenceConfig=inference_config
    )
    
    generated_text = response['output']['message']['content'][0]['text']
    print(generated_text.strip())

except Exception as e:
    print(f"Error: {e}")

Calling Model: google.gemma-3-4b-it
----------------------------------------
Artificial intelligence is rapidly transforming healthcare, offering a multitude of benefits. AI-powered tools can analyze vast amounts of patient data to improve diagnostic accuracy, personalize treatment plans, and predict potential health risks with greater precision. From assisting surgeons with robotic procedures to streamlining administrative tasks and accelerating drug discovery, AI is helping to reduce costs, improve patient outcomes, and ultimately, create a more efficient and effective healthcare system.
